# GS2 KBM database: how many samples are converged kinetic ballooning modes
The four `kbm_free` GS2 Latin hypercubes (SPR-045, M1, R4, R3). Each sample's dominant mode is fingerprinted with the user's
consensus classifier (`general_analysis.mode_classification.classify`): **KBM** if $\omega>0$ (ion direction), tearing parameter
$T<0.05$ (ballooning parity) and $0.25<\chi_e/\chi_i<4$, **MTM** if $\omega<0$ and $T>0.15$, otherwise **mixed**.
**Converged:** `growth_rate_tolerance` $<0.05$. This is the population used by the KBM paper's TGLF/GFTM comparisons (738 pooled).

In [1]:
import os
from multiprocessing import get_context

import numpy as np
import pandas as pd
from pyrokinetics import PyroHypercube
from general_analysis import mode_classification as mc

data_root = os.path.expandvars("$WORK/fwatts/Gyrokinetic_Simulations/GS2/Runs/LATIN_HYPERCUBE")
print("T_BAL", mc.T_BAL, "T_TEAR", mc.T_TEAR, "CHI_KBM", mc.CHI_KBM)

T_BAL 0.05 T_TEAR 0.15 CHI_KBM (0.25, 4.0)


In [2]:
def tied_species(sample):
    # sample_pyro does not re-apply the deck's derived settings (kbm_free pyroscan.json stores none), so tie the species as
    # GyroRun built the decks: ions share ion1's a/L_T, every a/L_n is the electron's, quasineutral.
    species = gs2_scan.sample_pyro(sample).local_species
    ions = [name for name in species.names if name != "electron"]
    for ion in ions[1:]:
        species[ion].inverse_lt = species[ions[0]].inverse_lt
    for ion in ions:
        species[ion].inverse_ln = species["electron"].inverse_ln
    species.enforce_quasineutrality("electron")
    return {name: (float(species[name].temp.m), float(species[name].dens.m), float(species[name].inverse_lt.m)) for name in species.names}


rows = []
for case in ["SPR-045", "M1", "R4", "R3"]:
    leaf = f"{data_root}/{case}/kbm_free"
    gs2_scan = PyroHypercube(pyroscan_json=f"{leaf}/pyroscan.json", load_base_pyro=True)
    gs2_scan.load_gk_output(netcdf_file=f"{leaf}/pyroscan.nc")
    gs2_output = gs2_scan.gk_output.data
    n_samples = gs2_output.sizes["sample"]
    with get_context("fork").Pool(int(os.environ.get("SLURM_CPUS_PER_TASK", 4))) as pool:
        species = pool.map(tied_species, range(n_samples), chunksize=8)
    heat = gs2_output.heat.sum("field").pint.dequantify()
    particle = gs2_output.particle.sum("field").pint.dequantify()

    def chi(group):   # (Q_s - 1.5 T_s Gamma_s) / (n_s T_s a/L_Ts), summed over the group
        numerator = denominator = 0.0
        for name in group:
            temp, dens, inverse_lt = np.array([sp[name] for sp in species]).T
            numerator = numerator + heat.sel(species=name).values - 1.5 * temp * particle.sel(species=name).values
            denominator = denominator + dens * temp * inverse_lt
        return numerator / denominator

    ions = [str(name) for name in gs2_output.species.values if name != "electron"]
    with np.errstate(divide="ignore", invalid="ignore"):
        labels = np.asarray(mc.classify(dict(T=gs2_output.tearing_parameter.values.ravel(),
                                             omega=mc.ION_DIRECTION * gs2_output.mode_frequency.pint.dequantify().values.ravel(),
                                             chi_ratio=chi(["electron"]) / chi(ions))))
    gamma = gs2_output.growth_rate.pint.dequantify().values.ravel()
    tolerance = gs2_output.growth_rate_tolerance.pint.dequantify().values.ravel()
    unstable = np.isfinite(gamma) & (gamma > 0)
    converged = unstable & (tolerance < 0.05)
    rows.append(dict(case=case, samples=n_samples, unstable=unstable.sum(), converged=converged.sum(),
                     KBM=(unstable & (labels == "KBM")).sum(), KBM_converged=(converged & (labels == "KBM")).sum(),
                     MTM_converged=(converged & (labels == "MTM")).sum(), mixed_converged=(converged & (labels == "mixed")).sum()))

population = pd.DataFrame(rows).set_index("case")
population.loc["total"] = population.sum()
population

,samples,unstable,converged,KBM,KBM_converged,MTM_converged,mixed_converged
case,,,,,,,
SPR-045,1000,645,530,228,203,54,273
M1,1000,718,616,155,140,107,369
R4,1000,802,696,220,208,229,259
R3,1000,857,736,202,187,245,304
total,4000,3022,2578,805,738,635,1205


`KBM_converged` is the paper's KBM database: the samples every TGLF/GFTM comparison uses.
`KBM` counts unstable KBMs whatever their convergence. `MTM_converged + mixed_converged + KBM_converged = converged`.

In [3]:
assert (population.KBM_converged + population.MTM_converged + population.mixed_converged == population.converged).all()
print(population.to_string())

         samples  unstable  converged  KBM  KBM_converged  MTM_converged  mixed_converged
case                                                                                     
SPR-045     1000       645        530  228            203             54              273
M1          1000       718        616  155            140            107              369
R4          1000       802        696  220            208            229              259
R3          1000       857        736  202            187            245              304
total       4000      3022       2578  805            738            635             1205
